# Article 1 — Stage-1 literature and Stage-2 data audit

Run this only after `00_colab_setup.ipynb`. It downloads a small HotpotQA distractor sample to inspect schema and records factual runtime metadata. It does **not** fit a model and it does **not** create publication claims.

In [ ]:
from pathlib import Path
import hashlib, json, platform, subprocess, sys

# The repository should already be cloned by 00_colab_setup.ipynb.
%cd /content/rag-evidence-robustness
SAMPLE_SIZE = 200
WRITE_TO_DRIVE = False
DRIVE_ROOT = Path('/content/drive/MyDrive/llm-rag-research/article1')
!{sys.executable} -m pip install -e '.[pilot]'


In [ ]:
from datasets import load_dataset

dataset = load_dataset('hotpotqa/hotpot_qa', 'distractor', split=f'train[:{SAMPLE_SIZE}]')
print('Rows:', len(dataset))
print('Fingerprint:', dataset._fingerprint)
print('Features:', dataset.features)
print('First record keys:', sorted(dataset[0].keys()))
print('First question:', dataset[0]['question'])
print('First context type:', type(dataset[0]['context']).__name__)


In [ ]:
# This is a metadata report, not a substitute for a pinned dataset revision.
sample_ids = [str(row.get('id', index)) for index, row in enumerate(dataset)]
report = {
    'dataset': 'hotpotqa/hotpot_qa',
    'config': 'distractor',
    'sample_size': len(dataset),
    'fingerprint': dataset._fingerprint,
    'features_repr': repr(dataset.features),
    'sample_id_sha256': hashlib.sha256('\n'.join(sample_ids).encode('utf-8')).hexdigest(),
    'python': sys.version,
    'platform': platform.platform(),
}
print(json.dumps(report, ensure_ascii=False, indent=2, sort_keys=True))
if WRITE_TO_DRIVE:
    out = DRIVE_ROOT / 'reports' / 'stage2_hotpot_schema_audit.json'
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(report, ensure_ascii=False, indent=2, sort_keys=True), encoding='utf-8')
    print('Wrote:', out)


## Required manual work before modelling

1. Read the original method and limitations sections of the P0 papers, then complete rows marked `full-method audit pending` in `docs/literature_matrix.csv`.
2. Inspect 10–20 downloaded records to confirm the context and supporting-fact fields match the assumptions in `docs/experiment_protocol.md`.
3. Send back this notebook's JSON report, the output schema, and any download/access errors. The next implementation step can then normalise the actual schema rather than guessing it.